# 01 - Explore the source data

Welcome! We are starting as data engineers by understanding the input before writing transformations. Real pipelines begin with questions about the source: what does each row mean, what fields exist, and where might quality problems appear?

This file is synthetic practice data. A row represents one order line. The dataset intentionally includes a few data quality issues for us to discover.

## Step 1 - Load the CSV

We use pandas in this first notebook because it makes a small dataset easy to inspect. Later, we can replace the transformation step with PySpark to show how the same ideas scale to larger data.

In [1]:
from pathlib import Path
import pandas as pd

# This works whether Jupyter starts in the project folder or in notebooks/.
candidates = [
    Path.cwd() / 'data' / 'source' / 'orders.csv',
    Path.cwd().parent / 'data' / 'source' / 'orders.csv',
]
source_file = next((path for path in candidates if path.exists()), None)
if source_file is None:
    raise FileNotFoundError('Could not find data/source/orders.csv. Start Jupyter from the project folder or notebooks folder.')

orders = pd.read_csv(source_file)
print(f'Loaded {len(orders)} rows from {source_file}')
orders.head()

Loaded 20 rows from C:\Users\taman\Documents\Codex\2026-10-10\lo\outputs\enterprise-cloud-data-integration-platform\data\source\orders.csv


,order_id,order_timestamp,customer_id,store_id,category,quantity,unit_price
0,1001,2026-09-01T09:15:00,501.0,store_north,books,2,12.5
1,1002,2026-09-01T10:04:00,502.0,store_north,stationery,5,3.2
2,1003,2026-09-01T11:27:00,503.0,store_south,books,1,18.0
3,1004,2026-09-01T12:03:00,504.0,store_east,home,2,25.0
4,1005,2026-09-01T13:46:00,505.0,store_south,stationery,3,4.5


## Step 2 - Understand the columns

Before transforming data, inspect its size, field types, missing values, and duplicate order IDs. These checks help us turn assumptions into an explicit data contract.

In [2]:
print('Shape (rows, columns):', orders.shape)
print('\nColumn types and non-null counts:')
orders.info()
print('\nMissing values per column:')
display(orders.isna().sum())
print('Duplicate order IDs:', orders['order_id'].duplicated().sum())

Shape (rows, columns): (20, 7)

Column types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         20 non-null     int64  
 1   order_timestamp  20 non-null     str    
 2   customer_id      19 non-null     float64
 3   store_id         20 non-null     str    
 4   category         20 non-null     str    
 5   quantity         20 non-null     int64  
 6   unit_price       20 non-null     float64
dtypes: float64(2), int64(2), str(3)
memory usage: 1.9 KB

Missing values per column:


order_id           0
order_timestamp    0
customer_id        1
store_id           0
category           0
quantity           0
unit_price         0
dtype: int64

Duplicate order IDs: 1


## What we learned

Look for a missing customer ID, a zero quantity, a negative unit price, and a repeated order ID. In the next step we will write down the rules for these fields, preserve the original rows, and send invalid records to quarantine with clear reasons instead of silently deleting them.

**Why inspect first?** If we transform blindly, bad assumptions can turn into bad reports. Profiling the source is a small step that prevents bigger downstream errors.